![SGSSS Logo](https://github.com/SGSSSonline/text-analysis-for-social-scientists/blob/main/img/SGSSS_Stacked.png?raw=true)

# Practical 2: Descriptive Text Analysis

In the previous practical, we learned how to transform raw text into a clean, structured format. In this practical, we put that preprocessed text to work. We explore a range of descriptive text analysis techniques that allow us to characterise the content and patterns within a corpus — from simple word counts to more sophisticated measures of association and comparison.

These methods are foundational: they are often the first step in any text analysis project, helping researchers understand what is in their data before moving on to more complex modelling techniques.

## Aims

This practical has two aims:

1. **Demonstrate how to use R and quanteda to perform descriptive text analysis** on social science data.
2. **Cultivate computational thinking skills** — learning to summarise, visualise, and compare textual data using a range of techniques.

### Lesson Details

| | |
| --- | --- |
| **Level** | Introductory |
| **Time** | ~45 minutes |
| **Pre-requisites** | Practical 1: Preprocessing Text |
| **Learning outcomes** | Compute and visualise word frequencies |
| | Generate and interpret word clouds, including comparative word clouds |
| | Use Keywords-in-Context (KWIC) to examine how terms are used |
| | Identify collocations and interpret co-occurrence patterns |
| | Compare vocabulary across groups within a corpus |

## Guide to Using This Resource

This is a **Jupyter Notebook** — an interactive document that combines text, code, and output in a single environment. If you are viewing this in **Google Colab**, you are running the notebook in the cloud, which means you do not need to install anything on your own machine.

**Note: This notebook uses R.** In Google Colab, you need to change the runtime: go to **Runtime > Change runtime type > select R**.

A notebook is made up of **cells**. There are two main types:

- **Markdown cells** contain formatted text (like this one). They provide explanations, instructions, and context.
- **Code cells** contain R code that you can execute. Code cells are displayed with a grey background and have a play button on the left.

To **run a cell**, click on it and press `Shift+Enter` (or click the play button). The output will appear directly below the cell. You should run the code cells **in order**, from top to bottom, as later cells often depend on variables or packages loaded in earlier cells.

If you are using **Google Colab**, make sure to save a copy of this notebook to your own Google Drive first: go to **File > Save a copy in Drive**.

If you are new to Jupyter Notebooks and would like a more detailed introduction, see the excellent materials by Dani Arribas-Bel: [https://github.com/darribas/gds19/blob/master/content/labs/lab_00.ipynb](https://github.com/darribas/gds19/blob/master/content/labs/lab_00.ipynb)

### Interactive Test

Let's make sure everything is working. Run the cell below and enter your name when prompted.

In [ ]:
cat("Hello! What is your name?\n")
name <- readline()
cat(paste0("Welcome to Practical 2, ", name, "!\n"))

## Setup

Before we begin, we need to install and load the R packages we will use throughout this practical. These include:

- **quanteda** — a comprehensive package for managing and analysing text data. It provides functions for tokenisation, concordancing (KWIC), and constructing document-feature matrices.
- **quanteda.textstats** — for computing text statistics such as term frequencies, collocations, and lexical diversity.
- **quanteda.textplots** — for visualising text data, including word clouds.
- **ggplot2** — for creating high-quality statistical graphics.
- **dplyr** — for data manipulation.

In [ ]:
# Install packages (if needed on Colab)
install.packages(c("quanteda", "quanteda.textstats", "quanteda.textplots", "ggplot2", "dplyr"), quiet = TRUE)

library(quanteda)
library(quanteda.textstats)
library(quanteda.textplots)
library(ggplot2)
library(dplyr)

cat("Successfully loaded packages\n")

## Loading the Corpus

We will work with the same set of **synthetic** UK parliamentary speeches used in Practical 1. The speakers and speeches are fictional, designed to reflect realistic variation in policy topics and party positions.

In [ ]:
# Create a sample dataset of parliamentary speech excerpts
df <- data.frame(
  date = c(
    "2024-01-15", "2024-01-15", "2024-01-22", "2024-01-22", "2024-02-05",
    "2024-02-05", "2024-02-12", "2024-02-12", "2024-02-19", "2024-02-19",
    "2024-03-04", "2024-03-04", "2024-03-11", "2024-03-11", "2024-03-18",
    "2024-03-18", "2024-03-25", "2024-03-25", "2024-04-01", "2024-04-01"
  ),
  speaker = c(
    "Margaret Thornton", "James Whitfield", "Sarah Okonkwo", "David Hargreaves",
    "Emily Chen", "Robert MacLeod", "Fatima Hussain", "Thomas Brennan",
    "Catherine Lloyd", "Andrew Patel", "Helen Murray", "Kwame Asante",
    "Patricia Gallagher", "Ravi Sharma", "Fiona Blackwood", "Marcus Johnson",
    "Alison Crawford", "Yusuf Ali", "Diana Novak", "Christopher Reeves"
  ),
  party = c(
    "Labour", "Conservative", "Labour", "Conservative", "Labour",
    "SNP", "Labour", "Conservative", "Liberal Democrat", "Labour",
    "SNP", "Labour", "Conservative", "Labour", "SNP",
    "Conservative", "Labour", "Labour", "Liberal Democrat", "Conservative"
  ),
  speech_text = c(
    "The National Health Service remains the cornerstone of our social contract with the British people. We must invest in frontline services and address the chronic staffing shortages that are putting patients at risk. The waiting lists have grown to record levels and this government must take urgent action to reduce them.",
    "Economic growth is the foundation upon which all public services depend. By cutting taxes and reducing unnecessary regulation, we can unleash the potential of British businesses. The private sector, not the state, is the engine of prosperity and job creation in this country.",
    "The climate crisis demands immediate and ambitious action from this House. We cannot afford to delay investment in renewable energy infrastructure. Our children and grandchildren will judge us by the decisions we make today on carbon emissions and environmental protection.",
    "Immigration policy must be fair but firm. We need a points-based system that attracts the skilled workers our economy needs while maintaining control of our borders. The British people voted for sovereignty and we must deliver on that promise.",
    "Education is the great equaliser in our society. Every child, regardless of their background or postcode, deserves access to excellent teaching and modern facilities. We must close the attainment gap between the most and least disadvantaged pupils in our schools.",
    "Scotland's interests are consistently overlooked by this Westminster government. The devolution settlement is being undermined at every turn. The people of Scotland deserve the right to determine their own future and to have their voice heard on matters that affect their daily lives.",
    "The housing crisis is devastating communities across the country. Young people cannot afford to buy their first home and rents are consuming an ever larger share of household incomes. We need a massive programme of social housing construction to address this emergency.",
    "Defence spending must remain a top priority for this government. The threats we face from hostile state actors are growing more complex and more dangerous. We must ensure our armed forces have the equipment and resources they need to keep this nation safe.",
    "Mental health services are chronically underfunded and overstretched. Too many people are waiting months or even years for treatment they desperately need. Parity of esteem between physical and mental health must become a reality, not just a slogan.",
    "The cost of living crisis is hitting working families hardest. Energy bills have skyrocketed and food prices continue to rise at alarming rates. The government must do more to support those who are struggling to heat their homes and feed their children.",
    "The fishing communities of Scotland have been betrayed by broken promises on Brexit. Access to our waters and fair quota allocations are essential for the survival of these coastal communities. We will continue to fight for the rights of Scottish fishermen in this House.",
    "Public transport infrastructure is failing passengers across the north of England. Years of underinvestment have left communities isolated and reliant on expensive and unreliable services. We need a transport revolution that connects people to jobs, education, and opportunity.",
    "Law and order must be restored to our streets. Violent crime and antisocial behaviour are blighting communities and the police need the resources and powers to tackle these problems effectively. We will always stand on the side of victims and law-abiding citizens.",
    "The arts and creative industries contribute billions to our economy and enrich the cultural life of this nation. Cuts to arts funding have devastated regional theatres, museums, and music venues. Investment in culture is not a luxury; it is an investment in our communities and our identity.",
    "Rural communities in Scotland face unique challenges that this government fails to understand. Broadband connectivity, access to healthcare, and affordable transport are not luxuries but necessities. The centralisation of services in urban areas is leaving rural Scotland behind.",
    "Free trade agreements are essential for our post-Brexit economic strategy. British goods and services must have access to global markets on the most favourable terms possible. We are building new trading relationships that will benefit every region of the United Kingdom.",
    "Workers' rights must be strengthened, not eroded, in the modern economy. The rise of zero-hours contracts and the gig economy has created a new class of precarious employment. Every worker deserves fair pay, decent conditions, and the security of knowing their rights are protected by law.",
    "Child poverty is a stain on our national conscience. Over four million children in this country are growing up in poverty, and the numbers are rising. We must reform the welfare system to ensure that no child goes hungry and every family has a decent standard of living.",
    "Local government is the backbone of democracy in this country. Councils are being asked to do more with less, and essential services are being cut to the bone. Proper funding for local authorities is not just desirable; it is essential for the health of our democratic institutions.",
    "Science and innovation are the keys to our future prosperity. The United Kingdom has world-leading universities and research institutions that must be supported and celebrated. Government investment in research and development is an investment in the jobs and industries of tomorrow."
  ),
  stringsAsFactors = FALSE
)

cat(paste0("Dataset shape: ", nrow(df), " speeches, ", ncol(df), " columns\n"))
head(df)

Next, we apply the preprocessing steps from Practical 1. We create a `quanteda` corpus, tokenise the texts, remove punctuation and numbers, convert to lowercase, remove stopwords, and build a document-feature matrix (DFM). The DFM is the core data structure for most of the analyses that follow.

In [ ]:
# Create a quanteda corpus with metadata
corp <- corpus(df, text_field = "speech_text")

# Tokenise and preprocess
toks <- corp |>
  tokens(remove_punct = TRUE, remove_numbers = TRUE) |>
  tokens_tolower() |>
  tokens_remove(stopwords("en"))

# Build the document-feature matrix
my_dfm <- dfm(toks)

cat(paste0("Corpus: ", ndoc(corp), " documents\n"))
cat(paste0("DFM dimensions: ", nrow(my_dfm), " documents x ", ncol(my_dfm), " features\n"))
my_dfm

## Word Frequency Analysis

The simplest form of text analysis is counting words. Word frequencies tell us which terms dominate the corpus — they are often the first thing we examine in any text analysis project.

In `quanteda`, the `textstat_frequency()` function computes word frequencies from a DFM. It returns a data frame with the feature name, its frequency, rank, and the number of documents in which it appears.

In [ ]:
# Compute word frequencies across the entire corpus
freq <- textstat_frequency(my_dfm, n = 20)

cat("Top 20 most frequent words in the corpus:\n")
print(freq)

A bar chart makes the distribution easier to interpret visually. Let's plot the top 20 words using `ggplot2`.

In [ ]:
# Plot the top 20 most frequent words
freq$feature <- reorder(freq$feature, freq$frequency)

ggplot(freq, aes(x = feature, y = frequency)) +
  geom_col(fill = "steelblue") +
  coord_flip() +
  labs(
    title = "Top 20 Most Frequent Words in the Hansard Corpus",
    x = "Word",
    y = "Frequency"
  ) +
  theme_minimal()

Word frequencies tell us which terms dominate the corpus, but the most frequent words are not always the most informative. Words like "must", "government", and "services" appear frequently because they are common in parliamentary discourse generally — they do not necessarily distinguish one speech from another. We will return to this issue when we compare vocabulary across groups later in the practical.

## Word Clouds

A word cloud is a popular visualisation that displays words in sizes proportional to their frequency. While they are intuitive and visually appealing, they should be used with care: the size of words can be misleading, and they provide no information about context or relationships between words.

The `quanteda.textplots` package provides `textplot_wordcloud()`, which generates a word cloud directly from a DFM. Let's generate one for the entire corpus.

In [ ]:
# Generate a word cloud for the full corpus
textplot_wordcloud(my_dfm, max_words = 100, color = "steelblue")

### Comparative Word Clouds

Word clouds become more analytically useful when we compare across groups. The `quanteda` function `dfm_group()` aggregates the DFM by a grouping variable — in our case, party. When we pass the grouped DFM to `textplot_wordcloud()` with `comparison = TRUE`, it generates a comparative word cloud showing the most frequent terms for each group.

Let's compare the vocabulary across all four parties in our corpus.

In [ ]:
# Group the DFM by party
dfm_party <- dfm_group(my_dfm, groups = party)

# Generate a comparative word cloud
textplot_wordcloud(
  dfm_party,
  comparison = TRUE,
  max_words = 75,
  color = c("#0087DC", "#DC241f", "#FDBB30", "#FDF38E")
)

The comparative word cloud shows which terms are most distinctive to each party. Notice how each quadrant highlights different vocabulary priorities — for example, SNP speeches emphasise "scotland" and "communities", while Conservative speeches feature "british" and "trade".

### Strengths and Limitations of Word Clouds

**Strengths:**
- Intuitive and visually engaging — useful for presentations and exploratory analysis.
- Quickly communicate the dominant vocabulary of a corpus.

**Limitations:**
- Word size can be misleading — small differences in frequency may appear as large visual differences.
- No information about context — we cannot tell *how* a word is used, only that it appears frequently.
- Placement is random — spatial position carries no meaning.
- Can give a false sense of analytical rigour when used alone.

Word clouds are best used as a complement to other methods, not as a standalone analytical tool.

## Keywords-in-Context (KWIC)

Word frequency tells us *how often* a term appears, but not *how* it is used. Keywords-in-Context (KWIC), also known as **concordancing**, shows us the surrounding words each time a target term appears. This is crucial for understanding meaning in context.

KWIC analysis is one of `quanteda`'s particular strengths. The `kwic()` function operates directly on a tokens object and returns a structured data frame showing the left context, keyword, and right context for each match. This makes it easy to inspect, filter, and export concordance results.

Let's examine how the term "government" is used across the corpus.

In [ ]:
# KWIC for "government" — note: we use the original tokens (before stopword removal)
# so that the context windows contain all words for readability
toks_full <- tokens(corp, remove_punct = TRUE, remove_numbers = TRUE) |>
  tokens_tolower()

kw_government <- kwic(toks_full, pattern = "government", window = 5)

cat(paste0("Found ", nrow(kw_government), " occurrences of 'government'\n\n"))
print(kw_government)

The output shows each occurrence of "government" along with the five words before and after it. The `docname` column tells us which speech the occurrence comes from.

Let's also look at the terms "services" and "economy".

In [ ]:
# KWIC for "services"
kw_services <- kwic(toks_full, pattern = "services", window = 5)

cat(paste0("Found ", nrow(kw_services), " occurrences of 'services'\n\n"))
print(kw_services)

In [ ]:
# KWIC for "economy"
kw_economy <- kwic(toks_full, pattern = "economy", window = 5)

cat(paste0("Found ", nrow(kw_economy), " occurrences of 'economy'\n\n"))
print(kw_economy)

KWIC reveals *how* terms are used, not just how often — crucial for understanding meaning in context. For example, "services" might appear in discussions of the NHS, public transport, or defence, and only KWIC reveals which sense is intended in each case.

### Pattern Matching with KWIC

The `kwic()` function also supports **glob** and **regex** pattern matching. This is useful when you want to find all forms of a word (e.g., "invest", "investment", "investing") or search for phrases.

Let's search for all words beginning with "invest" using a glob pattern.

In [ ]:
# KWIC with glob pattern — find all words starting with "invest"
kw_invest <- kwic(toks_full, pattern = "invest*", valuetype = "glob", window = 5)

cat(paste0("Found ", nrow(kw_invest), " matches for 'invest*'\n\n"))
print(kw_invest)

We can also search for **multi-word phrases** by passing a `phrase()` object. Let's search for the phrase "public services".

In [ ]:
# KWIC for a multi-word phrase
kw_phrase <- kwic(toks_full, pattern = phrase("public services"), window = 5)

cat(paste0("Found ", nrow(kw_phrase), " matches for 'public services'\n\n"))
print(kw_phrase)

This programmatic access to concordance results is one of `quanteda`'s strengths. Because `kwic()` returns a data frame, you can easily filter results by document, speaker, or party — for example, to compare how different parties use a specific term.

## Collocations

Individual word frequencies tell us what terms are common, but they miss an important feature of language: words do not appear in isolation. **Collocations** are word pairs (or longer sequences) that occur together more than we would expect by chance. Identifying collocations helps us understand the *phrases* and *concepts* that characterise a corpus, not just isolated terms.

For example, "climate" and "crisis" appearing together frequently tells us something different from each word appearing independently.

The `quanteda.textstats` package provides `textstat_collocations()`, which identifies statistically significant multi-word expressions using a lambda statistic (related to Pointwise Mutual Information). We apply it to our tokens object and filter by a minimum count.

In [ ]:
# Find collocations (bigrams and trigrams) with a minimum count of 2
collocations <- textstat_collocations(toks, min_count = 2, size = 2:3)

cat("Top collocations by statistical significance:\n\n")
print(head(collocations, 20))

The `lambda` column indicates the strength of association between the words — higher values mean the words co-occur much more often than chance would predict. The `z` column is a z-score indicating statistical significance.

Collocations identify conceptually meaningful phrases such as political terms, policy areas, and rhetorical constructions that simple frequency counts might miss. Note that `textstat_collocations()` uses a more sophisticated statistical approach than simply counting bigram frequencies — it accounts for the individual frequencies of each word to identify pairs that are genuinely associated.

## Comparing Vocabulary Across Groups

One of the most powerful applications of descriptive text analysis in the social sciences is comparing language use across groups. In our Hansard data, the natural grouping variable is **party**. Do different parties use different vocabularies? Are some terms more dominant in one party's speeches than another's?

Let's compare the speeches of the four parties in our corpus.

In [ ]:
# Check the distribution of speeches by party
cat("Number of speeches by party:\n")
table(docvars(my_dfm, "party"))

### Top Terms by Party

Let's find the 10 most frequent terms for each party using `topfeatures()` applied to the grouped DFM.

In [ ]:
# Group the DFM by party and display top features for each
dfm_party <- dfm_group(my_dfm, groups = party)

parties <- sort(rownames(dfm_party))

for (p in parties) {
  cat(paste0("\n", p, ":\n"))
  top <- topfeatures(dfm_party[p, ], n = 10)
  for (i in seq_along(top)) {
    cat(sprintf("  %-20s %d\n", names(top)[i], top[i]))
  }
}

We can also visualise this comparison using faceted bar charts with `ggplot2`.

In [ ]:
# Compute word frequencies grouped by party
freq_party <- textstat_frequency(my_dfm, n = 10, groups = party)

# Define party colours
party_colours <- c(
  "Conservative" = "#0087DC",
  "Labour" = "#DC241f",
  "Liberal Democrat" = "#FDBB30",
  "SNP" = "#FDF38E"
)

# Plot as faceted horizontal bar charts
ggplot(freq_party, aes(x = reorder_within(feature, frequency, group), y = frequency, fill = group)) +
  geom_col(show.legend = FALSE) +
  facet_wrap(~ group, scales = "free_y") +
  coord_flip() +
  scale_x_reordered() +
  scale_fill_manual(values = party_colours) +
  labs(
    title = "Top 10 Terms by Party",
    x = "Word",
    y = "Frequency"
  ) +
  theme_minimal()

Note: The `reorder_within()` and `scale_x_reordered()` functions come from the `tidytext` package. If they are not available, you can install `tidytext` or simply remove the reordering — the chart will still work but the bars may not be sorted within each facet. Alternatively, `quanteda` provides a simpler approach with `textplot_wordcloud()` as we saw earlier.

### Type-Token Ratio (TTR)

The **type-token ratio** (TTR) is a simple measure of vocabulary diversity. It is calculated as:

$$\text{TTR} = \frac{\text{Number of unique words (types)}}{\text{Total number of words (tokens)}}$$

A higher TTR indicates greater vocabulary diversity. The `quanteda.textstats` package provides `textstat_lexdiv()` which computes TTR (and other lexical diversity measures) directly from a DFM.

Let's compare TTR across parties.

In [ ]:
# Compute TTR for the grouped DFM (one row per party)
dfm_party <- dfm_group(my_dfm, groups = party)
ttr <- textstat_lexdiv(dfm_party, measure = "TTR")

cat("Vocabulary diversity (TTR) by party:\n\n")
print(ttr)

Note that TTR is sensitive to text length — shorter texts tend to have higher TTR values because there are fewer opportunities for words to repeat. When comparing groups, be mindful of differences in the total number of tokens. In our sample, the parties have different numbers of speeches (and therefore different total token counts), which affects the TTR comparison.

Despite this caveat, TTR provides a useful first approximation of vocabulary diversity and can flag interesting differences for further investigation.

## Exercise

Choose a research question you can explore with the tools introduced in this practical. For example:

- How do different parties talk about the NHS?
- What are the most distinctive terms used by each speaker?
- Are there notable collocations that differ between Conservative and Labour speeches?

Apply **at least two** of the methods above (word frequencies, word clouds, KWIC, collocations, or comparative analysis) to investigate your question.

Use the skeleton code below as a starting point. **Note**: You can use your own text data if you prefer.

In [ ]:
# State your research question
# Research question:
# YOUR RESEARCH QUESTION HERE

In [ ]:
# Method 1: Apply your first analytical method
# For example, word frequency analysis or KWIC for a specific term

# YOUR CODE HERE

In [ ]:
# Method 2: Apply your second analytical method
# For example, comparative word clouds or collocations

# YOUR CODE HERE

In [ ]:
# Interpretation: What did you find? Write your interpretation as comments below.

# YOUR INTERPRETATION HERE

## Appendix: Exercise Solution

Below is a worked example investigating the question: **How do Conservative and Labour MPs talk about the economy?**

We apply two methods: KWIC analysis and comparative word frequency.

In [ ]:
# --- Solution: Method 1 — KWIC for 'economy' by party ---

# Create tokens objects for each party (with stopwords retained for readable context)
toks_con <- corpus_subset(corp, party == "Conservative") |>
  tokens(remove_punct = TRUE, remove_numbers = TRUE) |>
  tokens_tolower()

toks_lab <- corpus_subset(corp, party == "Labour") |>
  tokens(remove_punct = TRUE, remove_numbers = TRUE) |>
  tokens_tolower()

cat("=" |> strrep(70), "\n")
cat("Conservative speeches — KWIC for 'economy':\n")
cat("=" |> strrep(70), "\n")
print(kwic(toks_con, pattern = "economy", window = 5))

cat("\n", "=" |> strrep(70), "\n")
cat("Labour speeches — KWIC for 'economy':\n")
cat("=" |> strrep(70), "\n")
print(kwic(toks_lab, pattern = "economy", window = 5))

In [ ]:
# --- Solution: Method 2 — Comparative word frequencies ---

# Create DFMs for Conservative and Labour (with stopwords removed)
dfm_con <- corpus_subset(corp, party == "Conservative") |>
  tokens(remove_punct = TRUE, remove_numbers = TRUE) |>
  tokens_tolower() |>
  tokens_remove(stopwords("en")) |>
  dfm()

dfm_lab <- corpus_subset(corp, party == "Labour") |>
  tokens(remove_punct = TRUE, remove_numbers = TRUE) |>
  tokens_tolower() |>
  tokens_remove(stopwords("en")) |>
  dfm()

# Get frequency counts for each party
freq_con <- colSums(dfm_con)
freq_lab <- colSums(dfm_lab)

# Combine into a single data frame
all_features <- union(names(freq_con), names(freq_lab))
comp_df <- data.frame(
  word = all_features,
  Conservative = sapply(all_features, function(w) ifelse(w %in% names(freq_con), freq_con[w], 0)),
  Labour = sapply(all_features, function(w) ifelse(w %in% names(freq_lab), freq_lab[w], 0)),
  stringsAsFactors = FALSE,
  row.names = NULL
)
comp_df$difference <- comp_df$Conservative - comp_df$Labour
comp_df <- comp_df[order(comp_df$difference), ]

cat("Words most skewed towards Labour (negative difference):\n")
print(head(comp_df, 10), row.names = FALSE)

cat("\nWords most skewed towards Conservative (positive difference):\n")
print(tail(comp_df, 10), row.names = FALSE)

In [ ]:
# --- Solution: Visualise the most distinctive terms ---

# Combine the most Labour-skewed and most Conservative-skewed terms
distinctive <- rbind(head(comp_df, 10), tail(comp_df, 10))
distinctive$colour <- ifelse(distinctive$difference < 0, "Labour", "Conservative")
distinctive$word <- factor(distinctive$word, levels = distinctive$word)

ggplot(distinctive, aes(x = word, y = difference, fill = colour)) +
  geom_col() +
  coord_flip() +
  scale_fill_manual(values = c("Conservative" = "#0087DC", "Labour" = "#DC241f")) +
  geom_hline(yintercept = 0, linewidth = 0.3) +
  labs(
    title = "Most Distinctive Words: Conservative vs Labour",
    x = "Word",
    y = "Frequency Difference (Conservative - Labour)",
    fill = "Party"
  ) +
  theme_minimal()

**Interpretation**: The KWIC analysis shows that Conservative MPs tend to discuss the "economy" in the context of growth, trade, and the private sector, while Labour MPs reference it in the context of workers' rights and the gig economy. The comparative frequency analysis reveals that Labour speeches are more likely to use terms related to social welfare (e.g., "poverty", "children", "rights"), while Conservative speeches emphasise terms related to trade, security, and sovereignty. These patterns align with well-documented differences in party priorities and framing strategies.

## Bibliography

- Benoit, K., Watanabe, K., Wang, H., Noel, P., Lori, S., Obeng, A., Muller, S., & Matsuo, A. (2018). quanteda: An R package for the quantitative analysis of textual data. *Journal of Open Source Software*, 3(30), 774.
- Grimmer, J., Roberts, M. E., & Stewart, B. M. (2022). *Text as Data: A New Framework for Machine Learning and the Social Sciences*. Princeton University Press.
- Manning, C. D., Raghavan, P., & Schutze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press.
- Sinclair, J. (1991). *Corpus, Concordance, Collocation*. Oxford University Press.
- Baker, P. (2006). *Using Corpora in Discourse Analysis*. Continuum.
- Church, K. W., & Hanks, P. (1990). Word association norms, mutual information, and lexicography. *Computational Linguistics*, 16(1), 22-29.

---

**END OF FILE**